# XTTS v2 Voice Clone — NBA Voiceover Worker

Colab worker for the NBA voiceover pipeline (see `docs/NBA_PIPELINE_SPEC.md`).

**What it does:** mounts Google Drive, installs coqui-tts, clones a voice from
`voice_sample.wav`, reads `script.txt` dropped by n8n, speaks the script in the
cloned voice, and writes `output.wav` back to Drive.

**Expected Drive layout:**
```
/content/drive/MyDrive/
├── voice_sample.wav          ← 10–30 s clean reference audio of the target voice
└── NBA_Voiceovers/
    ├── script.txt            ← narration text (dropped by n8n)
    └── output.wav            ← written by this notebook
```

**Runtime:** GPU recommended (Runtime → Change runtime type → T4). CPU works
but is roughly 10× slower per clip.

**Credentials:** none handled here. The notebook only touches the mounted
Drive of whichever Google account runs it.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT = '/content/drive/MyDrive'
VOICE_SAMPLE = os.path.join(DRIVE_ROOT, 'voice_sample.wav')
OUTPUT_DIR = os.path.join(DRIVE_ROOT, 'NBA_Voiceovers')
SCRIPT_PATH = os.path.join(OUTPUT_DIR, 'script.txt')
OUTPUT_WAV = os.path.join(OUTPUT_DIR, 'output.wav')

os.makedirs(OUTPUT_DIR, exist_ok=True)

assert os.path.exists(VOICE_SAMPLE), (
    f'No voice sample at {VOICE_SAMPLE}. Upload a 10-30 s clean recording '
    'of the target voice to MyDrive first.'
)
print('Drive mounted.')
print(f'  voice sample : {VOICE_SAMPLE}')
print(f'  output dir   : {OUTPUT_DIR}')

## 2. Install coqui-tts

Pinned to the maintained fork (`coqui-tts` on PyPI, the idiots-in-residence
continuation) because the original `TTS` package is unmaintained and breaks
on current Python. First install takes a few minutes.

In [ ]:
%pip install -q coqui-tts

# torch arrives with colab; make sure it sees the GPU before we pay for model download
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'torch {torch.__version__} — using {device}')
if device == 'cpu':
    print('WARNING: no GPU. Inference will be slow; consider Runtime > Change runtime type.')

## 3. Load XTTS v2 with the cloned voice

The model downloads (~1.8 GB) on first run and is cached for later sessions.

In [ ]:
import io
import contextlib
from TTS.api import TTS

# Agree to the CPML licence non-interactively (XTTS v2 requires it).
os.environ['COQUI_TOS_AGREED'] = '1'

tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
print('XTTS v2 loaded.')

## 4. Read the script

Reads `script.txt`. If it is not there yet, polls briefly — n8n drops the
file then triggers this worker, but the two events can race by seconds.

In [ ]:
import time

def read_script(path: str, timeout_s: int = 120) -> str:
    """Poll for script.txt so an n8n drop-and-trigger race cannot lose the run."""
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if os.path.exists(path):
            text = open(path, encoding='utf-8').read().strip()
            if text:
                return text
        print(f'waiting for {path} ...')
        time.sleep(5)
    raise FileNotFoundError(f'{path} did not appear within {timeout_s}s')

script = read_script(SCRIPT_PATH)
print(f'script ({len(script)} chars):\n')
print(script[:500] + ('...' if len(script) > 500 else ''))

## 5. Synthesize with the cloned voice

Long scripts are split into sentences and synthesized per-sentence, then
concatenated — XTTS degrades on very long single passes (drifting pace,
occasional truncation). Language is English; change `language=` if the
script is not.

In [ ]:
import re
import numpy as np
import soundfile as sf

def split_sentences(text: str, max_len: int = 250):
    """Split into sentence-ish chunks under max_len chars (XTTS comfort limit)."""
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks, buf = [], ''
    for p in parts:
        if buf and len(buf) + len(p) + 1 <= max_len:
            buf += ' ' + p
        else:
            if buf:
                chunks.append(buf)
            # A single over-long sentence gets hard-split at a comma boundary.
            while len(p) > max_len:
                cut = p.rfind(',', 0, max_len)
                cut = cut if cut > max_len // 2 else max_len
                chunks.append(p[:cut].rstrip(','))
                p = p[cut:].lstrip(', ')
            buf = p
    if buf:
        chunks.append(buf)
    return chunks

chunks = split_sentences(script)
print(f'{len(chunks)} chunk(s) to synthesize')

sample_rate = 24_000  # XTTS v2 native rate
pieces = []
for i, chunk in enumerate(chunks, 1):
    wav = tts.tts(
        text=chunk,
        speaker_wav=VOICE_SAMPLE,
        language='en',
        split_sentences=False,  # already chunked ourselves
    )
    pieces.append(np.array(wav, dtype=np.float32))
    # Small gap between chunks keeps sentence boundaries audible.
    pieces.append(np.zeros(int(sample_rate * 0.25), dtype=np.float32))
    print(f'  [{i}/{len(chunks)}] done')

full_audio = np.concatenate(pieces)

## 6. Export to Drive

In [ ]:
sf.write(OUTPUT_WAV, full_audio, sample_rate)

duration_s = len(full_audio) / sample_rate
size_mb = os.path.getsize(OUTPUT_WAV) / 1e6
print(f'wrote {OUTPUT_WAV}')
print(f'  duration: {duration_s:.1f}s  size: {size_mb:.1f} MB')

# Audible sanity check inside the notebook
from IPython.display import Audio
Audio(data=full_audio, rate=sample_rate)

## 7. (Optional) Watch mode

Run this cell instead of re-running 4–6 when n8n will trigger repeated runs:
it waits for each new `script.txt`, renders it, renames the script aside, and
loops. Stop with the notebook's interrupt button.

In [ ]:
# WATCH_MODE = True
# while WATCH_MODE:
#     try:
#         script = read_script(SCRIPT_PATH, timeout_s=3600)
#         chunks = split_sentences(script)
#         pieces = []
#         for chunk in chunks:
#             wav = tts.tts(text=chunk, speaker_wav=VOICE_SAMPLE,
#                           language='en', split_sentences=False)
#             pieces.append(np.array(wav, dtype=np.float32))
#             pieces.append(np.zeros(int(sample_rate * 0.25), dtype=np.float32))
#         sf.write(OUTPUT_WAV, np.concatenate(pieces), sample_rate)
#         os.rename(SCRIPT_PATH, SCRIPT_PATH + '.done-' + str(int(time.time())))
#         print('rendered one script; waiting for the next...')
#     except KeyboardInterrupt:
#         break